# FireBridge 2/3: Preprocessing
Profiles, explores and cleans the raw data from notebook 1, then saves one clean file per country.

**Input:** `FireBridge/data/raw/`  ·  **Output:** `FireBridge/data/processed/<country>_clean.parquet`

Both sensors are read with the same column names:

| MODIS (raw) | VIIRS (raw) | Here |
|---|---|---|
| `brightness` | `bright_ti4` | `bright_fire` |
| `bright_t31` | `bright_ti5` | `bright_bg` |
| `confidence` 0-100 | `confidence` l/n/h | `confidence` (VIIRS l/n/h → 20/60/90) + `confidence_cat` |
| `acq_date` + `acq_time` | same | `datetime` (UTC) |

In [ ]:
!pip -q install duckdb
from google.colab import drive
drive.mount('/content/drive')

## Functions
Run once per session.  
Data: `listDownloadedCountries`, `readCountries`.  
Profiling: `profileData`, `typeBreakdown`, `frpReport`, `monthlyCounts`.  
Plots: `plotFRP`, `plotByType`, `plotDayNight`, `plotMonthly`, `plotMap`.  
Cleaning: `cleanFireData`, `saveProcessed`.

In [ ]:
import os, duckdb, pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display


# ---------------- data access (same as the download notebook) ----------------
RAW_DIR = '/content/drive/MyDrive/FireBridge/data/raw'   # written by the download notebook
SENSORS = {'modis': {'first_year': 2000}, 'viirs': {'first_year': 2012}}

# ---------------- helpers (internal) ----------------
def _country_key(name):
    """'United States' -> 'United_States' (the spelling FIRMS uses in file names)."""
    return name.strip().replace(' ', '_')


def listDownloadedCountries():
    """Show what is already downloaded: one row per country and sensor."""
    rows = []
    if not os.path.isdir(RAW_DIR):
        return pd.DataFrame(columns=['country', 'sensor', 'first_year', 'last_year', 'years', 'size_mb'])
    for sdir in sorted(os.listdir(RAW_DIR)):
        if not sdir.startswith('sensor='): continue
        for cdir in sorted(os.listdir(f'{RAW_DIR}/{sdir}')):
            if not cdir.startswith('country='): continue
            years, size = [], 0
            for ydir in os.listdir(f'{RAW_DIR}/{sdir}/{cdir}'):
                f = f'{RAW_DIR}/{sdir}/{cdir}/{ydir}/data.parquet'
                if ydir.startswith('year=') and os.path.exists(f):
                    years.append(int(ydir[5:])); size += os.path.getsize(f)
            if years:
                rows.append({'country': cdir[8:], 'sensor': sdir[7:], 'first_year': min(years),
                             'last_year': max(years), 'years': len(years), 'size_mb': round(size / 1e6, 1)})
    return pd.DataFrame(rows).sort_values(['country', 'sensor']).reset_index(drop=True)


def readCountries(countries=None, sensors=('modis', 'viirs'), years=None,
                  vegetation_only=True, harmonize=True, output='pandas'):
    """Read downloaded data, loading only what you ask for.

    countries       : list of names (default: all downloaded)
    sensors         : 'modis' and/or 'viirs'
    years           : (start, end) inclusive, e.g. (2012, 2025); default all
    vegetation_only : keep only type == 0 (drops gas flares, industry, offshore)
    harmonize       : same column names for both sensors + numeric confidence + UTC datetime
    output          : 'pandas' -> DataFrame, 'duckdb' -> lazy DuckDB relation (for data too big for RAM)
    """
    parts = []
    for s in sensors:
        glob = f'{RAW_DIR}/sensor={s}/*/*/data.parquet'
        where = []
        if countries:
            names = ', '.join(f"'{_country_key(c)}'" for c in countries)
            where.append(f'country IN ({names})')
        if years:
            where.append(f'year BETWEEN {years[0]} AND {years[1]}')
        if vegetation_only:
            where.append('type = 0')
        w = ('WHERE ' + ' AND '.join(where)) if where else ''
        if harmonize:
            fire, bg = ('brightness', 'bright_t31') if s == 'modis' else ('bright_ti4', 'bright_ti5')
            conf = ('CAST(confidence AS SMALLINT)' if s == 'modis' else
                    "CAST(CASE confidence WHEN 'l' THEN 20 WHEN 'n' THEN 60 WHEN 'h' THEN 90 END AS SMALLINT)")
            conf_cat = ("CASE WHEN confidence < 30 THEN 'low' WHEN confidence < 80 THEN 'nominal' ELSE 'high' END"
                        if s == 'modis' else
                        "CASE confidence WHEN 'l' THEN 'low' WHEN 'n' THEN 'nominal' WHEN 'h' THEN 'high' END")
            sel = f"""latitude, longitude, {fire} AS bright_fire, {bg} AS bright_bg, scan, track, frp,
                      {conf} AS confidence, {conf_cat} AS confidence_cat,
                      acq_date + make_time(acq_time // 100, acq_time % 100, 0) AS datetime,
                      satellite, daynight, type, '{s}' AS sensor, country, CAST(year AS SMALLINT) AS year"""
        else:
            sel = '* REPLACE (CAST(confidence AS VARCHAR) AS confidence)'   # sensor, country, year come from the folder names
        if not any(True for _ in _glob_files(s)):
            continue
        parts.append(f"SELECT {sel} FROM read_parquet('{glob}', hive_partitioning=true) {w}")
    if not parts:
        print('Nothing downloaded yet for these sensors. Run downloadCountries first.')
        return None
    rel = duckdb.sql(' UNION ALL BY NAME '.join(parts))   # default connection, so duckdb.sql() can query the result
    return rel if output == 'duckdb' else rel.df()


def _glob_files(sensor):
    d = f'{RAW_DIR}/sensor={sensor}'
    if os.path.isdir(d):
        for root, _, files in os.walk(d):
            for f in files:
                if f == 'data.parquet':
                    yield f


# ---------------- preprocessing ----------------
PROCESSED_DIR = '/content/drive/MyDrive/FireBridge/data/processed'
TYPE_NAMES = {0: 'vegetation fire', 1: 'volcano', 2: 'static land source (gas flare / industry)', 3: 'offshore'}


def _by_sensor(df):
    for s in sorted(df['sensor'].unique()):
        yield s.upper(), df[df['sensor'] == s]


# ---------------- profiling ----------------
def profileData(df):
    """Shape, duplicates, date range, invalid coordinates, missing values, and numeric/categorical summaries per sensor."""
    num_cols = ['latitude', 'longitude', 'scan', 'track', 'frp', 'bright_fire', 'bright_bg']
    cat_cols = ['satellite', 'confidence_cat', 'daynight', 'type']
    for name, d in _by_sensor(df):
        print(f'\n========== {name}: {len(d):,} rows ==========')
        print('Countries:', ', '.join(sorted(d['country'].unique())))
        print('Date range:', d['datetime'].min(), '→', d['datetime'].max())
        print('Duplicates:', d.duplicated(subset=['latitude', 'longitude', 'datetime']).sum())
        print('Missing values:', {c: int(v) for c, v in d.isna().sum().items() if v})
        bad = (~d['latitude'].between(-90, 90) | ~d['longitude'].between(-180, 180)).sum()
        print('Invalid coordinates:', int(bad))
        display(d[num_cols].describe().round(2))
        for c in cat_cols:
            vc = d[c].value_counts(dropna=False)
            display(pd.DataFrame({'count': vc, 'pct': (vc / len(d) * 100).round(2)}).rename_axis(c))


def typeBreakdown(df):
    """How many detections of each type (vegetation fire, gas flare, ...) per sensor."""
    t = df.groupby(['sensor', 'type']).size().rename('count').reset_index()
    t['pct'] = (100 * t['count'] / t.groupby('sensor')['count'].transform('sum')).round(2)
    t['meaning'] = t['type'].map(TYPE_NAMES)
    return t


def frpReport(df):
    """Missing / zero / negative FRP and upper percentiles per sensor."""
    rows = []
    for name, d in _by_sensor(df):
        q = d['frp'].quantile([0.5, 0.9, 0.95, 0.99, 0.999])
        rows.append({'sensor': name, 'rows': len(d), 'missing': int(d['frp'].isna().sum()),
                     'zero': int((d['frp'] == 0).sum()), 'negative': int((d['frp'] < 0).sum()),
                     'median': q[0.5], 'p90': q[0.9], 'p95': q[0.95], 'p99': q[0.99], 'p99.9': q[0.999],
                     'max': d['frp'].max()})
    return pd.DataFrame(rows).round(2)


def monthlyCounts(df):
    """Detections per month (rows) and sensor (columns)."""
    return (df.groupby([df['datetime'].dt.to_period('M'), 'sensor']).size()
              .unstack(fill_value=0).rename_axis('month'))


# ---------------- plots ----------------
def plotFRP(df, max_frp=None):
    """FRP histograms (log scale, because a few huge fires dominate) and boxplots per sensor."""
    sensors = list(_by_sensor(df))
    fig, axes = plt.subplots(2, len(sensors), figsize=(7 * len(sensors), 8), squeeze=False)
    for i, (name, d) in enumerate(sensors):
        frp = d['frp'] if max_frp is None else d['frp'].clip(upper=max_frp)
        axes[0, i].hist(frp[frp > 0], bins=np.logspace(-1, np.log10(max(frp.max(), 1)), 60))
        axes[0, i].set_xscale('log'); axes[0, i].set_title(f'{name} FRP distribution'); axes[0, i].set_xlabel('FRP (MW, log scale)')
        sns.boxplot(x=frp[frp > 0], ax=axes[1, i], log_scale=True); axes[1, i].set_title(f'{name} FRP boxplot')
    plt.tight_layout(); plt.show()


def plotByType(df):
    """FRP by detection type: gas flares look very different from vegetation fires."""
    sensors = list(_by_sensor(df))
    fig, axes = plt.subplots(1, len(sensors), figsize=(7 * len(sensors), 5), squeeze=False)
    for i, (name, d) in enumerate(sensors):
        sns.boxplot(data=d[d['frp'] > 0], x='type', y='frp', ax=axes[0, i], log_scale=True)
        axes[0, i].set_title(f'{name} FRP by detection type')
    plt.tight_layout(); plt.show()


def plotDayNight(df):
    """Day vs night detections per sensor."""
    sensors = list(_by_sensor(df))
    fig, axes = plt.subplots(1, len(sensors), figsize=(5 * len(sensors), 4), squeeze=False)
    for i, (name, d) in enumerate(sensors):
        d['daynight'].value_counts().reindex(['D', 'N']).plot(kind='bar', ax=axes[0, i])
        axes[0, i].set_title(f'{name}: day vs night'); axes[0, i].set_ylabel('detections')
    plt.tight_layout(); plt.show()


def plotMonthly(df):
    """Monthly detections per sensor (the 2012 jump shows up here)."""
    m = monthlyCounts(df)
    ax = m.plot(figsize=(14, 4), logy=True)
    ax.set_title('Monthly detections'); ax.set_ylabel('detections (log scale)')
    plt.show()


def plotMap(df, max_points=200_000):
    """Where the detections are, per sensor (randomly sampled if there are too many points to draw)."""
    sensors = list(_by_sensor(df))
    fig, axes = plt.subplots(1, len(sensors), figsize=(7 * len(sensors), 6), squeeze=False)
    for i, (name, d) in enumerate(sensors):
        if len(d) > max_points:
            d = d.sample(max_points, random_state=0)
        axes[0, i].scatter(d['longitude'], d['latitude'], s=0.5, alpha=0.3)
        axes[0, i].set_title(f'{name} detection locations'); axes[0, i].set_xlabel('longitude'); axes[0, i].set_ylabel('latitude')
        axes[0, i].set_aspect('equal', adjustable='datalim')
    plt.tight_layout(); plt.show()


# ---------------- cleaning ----------------
def cleanFireData(df, vegetation_only=True, drop_zero_frp=False, min_confidence=None):
    """Remove rows that would mislead the analysis, and report how many each step removed.

    - invalid datetime or coordinates
    - non-vegetation detections (gas flares, volcanoes, offshore), unless vegetation_only=False
    - exact duplicate detections (same sensor, place and time)
    - optionally zero-FRP rows and low-confidence rows (MODIS 0-100; VIIRS l/n/h mapped to 20/60/90)
    """
    log, out = [], df.copy()
    def step(label, mask):
        nonlocal out
        removed = int((~mask).sum()); out = out[mask]
        log.append({'step': label, 'removed': removed, 'remaining': len(out)})
    log.append({'step': 'start', 'removed': 0, 'remaining': len(out)})
    step('invalid datetime', out['datetime'].notna())
    step('invalid coordinates', out['latitude'].between(-90, 90) & out['longitude'].between(-180, 180))
    if vegetation_only:
        step('not a vegetation fire (type != 0)', out['type'] == 0)
    step('duplicates', ~out.duplicated(subset=['sensor', 'latitude', 'longitude', 'datetime']))
    if drop_zero_frp:
        step('zero FRP', out['frp'] > 0)
    if min_confidence is not None:
        step(f'confidence < {min_confidence}', out['confidence'] >= min_confidence)
    display(pd.DataFrame(log))
    return out.reset_index(drop=True)


def saveProcessed(df, name):
    """Save a processed table to FireBridge/data/processed/<name>.parquet."""
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    path = f'{PROCESSED_DIR}/{name}.parquet'
    df.to_parquet(path, index=False)
    print(f'Saved {len(df):,} rows → {path} ({os.path.getsize(path) / 1e6:.1f} MB)')
    return path


In [ ]:
listDownloadedCountries()

## Load one country
All detection types (`vegetation_only=False`), so the profiling shows what gets filtered out. For big countries, work on one country at a time or limit years with `years=(2018, 2025)`.

In [ ]:
COUNTRY = 'Egypt_Nile_Delta'     # any name from listDownloadedCountries(): a study region or a whole country
raw = readCountries([COUNTRY], vegetation_only=False)
print(f'{len(raw):,} rows, {raw.memory_usage(deep=True).sum() / 1e6:.0f} MB in memory')

## Data profiling

In [ ]:
profileData(raw)

## Detection types
Only `type == 0` (vegetation fires) is kept. Types 2 and 3 are gas flares, industry and offshore platforms: they burn constantly and would distort the seasonal pattern and the sensor comparison.

In [ ]:
typeBreakdown(raw)

In [ ]:
plotByType(raw)

## Fire radiative power (FRP)

In [ ]:
frpReport(raw[raw['type'] == 0])

In [ ]:
plotFRP(raw[raw['type'] == 0])

## Time, day/night and location

In [ ]:
monthlyCounts(raw[raw['type'] == 0]).tail(24)

In [ ]:
plotMonthly(raw[raw['type'] == 0])

In [ ]:
plotDayNight(raw[raw['type'] == 0])

In [ ]:
plotMap(raw[raw['type'] == 0])

## Clean and save

In [ ]:
clean = cleanFireData(raw)
saveProcessed(clean, f'{_country_key(COUNTRY).lower()}_clean')

### All downloaded regions and countries
One at a time, so memory stays low.

In [ ]:
for c in listDownloadedCountries()['country'].unique():
    print(f'\n### {c}')
    df = cleanFireData(readCountries([c], vegetation_only=False))
    saveProcessed(df, f'{c.lower()}_clean')
    del df